In [ ]:
import os
import json
import shutil
from pathlib import Path
import yaml
import random

# 1. 경로 설정
origin_root = Path("origin_sample_dataset") # <-여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True)

# 2. YOLO 디렉토리 구조 생성 (test 추가)
dirs = [
    "images/train",
    "images/val",
    "images/test", # test 디렉토리 추가
    "labels/train",
    "labels/val",
    "labels/test" # test 디렉토리 추가
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 생성
class_mapping = {
    # (작물_코드, JSON_disease_값): class_id
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 3 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 4 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 5 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 6 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 7 = 7: '사과탄저병'
}

# 4. JSON → YOLO 레이블 변환 함수
def convert_label(json_path, img_filename_stem, img_width, img_height):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. 파일명에서 작물 정보 추출 (V006_80_0_00_01_01_25_0_b06_20201005_0002_S01_1.jpg)
    # _ 기준으로 5번째 위치 (인덱스 4)
    filename_parts = img_filename_stem.split('_')
    if len(filename_parts) > 4:
        crop_code = filename_parts[4] # '01' (배) 또는 '02' (사과)
    else:
        print(f"    ! Warning: Could not extract crop code from filename: {img_filename_stem}. Skipping this image.")
        return "" # 유효하지 않은 파일명이면 빈 문자열 반환

    # 2. JSON에서 질병 정보 추출
    # annotations['disease']는 정수형 값일 것으로 가정합니다.
    json_disease_value = data['annotations']['disease']

    # 3. class_mapping을 사용하여 최종 class_id 결정
    # 튜플 (작물_코드, JSON_disease_값)를 키로 사용하여 class_id 찾기
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key) # .get()을 사용하여 키가 없을 경우 None 반환
    
    if class_id is None:
        print(f"    ! Warning: No class_id mapping found for crop_code '{crop_code}' and JSON disease value '{json_disease_value}' in {img_filename_stem}. Skipping this image.")
        return "" # 매핑 정보가 없으면 빈 문자열 반환

    # 바운딩 박스 처리 (기존과 동일)
    bbox_lines = []
    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # YOLO 형식으로 정규화
        x_center = ((xtl + xbr) / 2) / img_width
        y_center = ((ytl + ybr) / 2) / img_height
        width = (xbr - xtl) / img_width
        height = (ybr - ytl) / img_height
        
        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

# 5. 데이터 처리 함수
def process_dataset(src_type, target_types):
    print(f"\nProcessing {src_type} data...")
    
    # 원천 데이터 폴더 탐색
    for folder in (origin_root / src_type).iterdir():
        if folder.name.startswith("[원천]"):
            # 이미지 파일 목록 가져오기
            img_files = list(folder.glob("*.jpg"))
            random.shuffle(img_files) # 무작위 섞기
            
            # 훈련 데이터는 전체를 사용
            if src_type == "Training":
                target_type = "train"
                for img_file in img_files:
                    process_image(img_file, folder, target_type)
            # 검증 데이터는 5:5로 분할
            elif src_type == "Validation":
                split_idx = len(img_files) // 2
                val_files = img_files[:split_idx]
                test_files = img_files[split_idx:]
                
                for img_file in val_files:
                    process_image(img_file, folder, "val")
                
                for img_file in test_files:
                    process_image(img_file, folder, "test")

# 5-1. 개별 이미지 처리 함수
def process_image(img_file, origin_folder, target_type):
    # 이미지 복사
    dest_dir = yolo_root / "images" / target_type
    shutil.copy(img_file, dest_dir / img_file.name)
    
    # 대응되는 JSON 파일 찾기
    json_path = None
    label_folder = origin_folder.name.replace("[원천]", "[라벨]")
    
    # 파일명 소문자 변환
    stem_lower = img_file.stem.lower() # 파일명에서 확장자 제외한 부분
    
    # JSON 경로 후보 생성 (소문자 확장자에 맞추어 탐색)
    label_dir = origin_root / origin_folder.parent.name / label_folder
    for candidate in label_dir.glob("*.json"):
        if candidate.stem.lower() == stem_lower:
            json_path = candidate
            break
    
    if not json_path:
        print(f"    ! JSON not found for {img_file.name}")
        return
    
    # JSON에서 이미지 크기 추출
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        img_width = json_data['description']['width']
        img_height = json_data['description']['height']
    except Exception as e:
        print(f"    ! Error reading JSON for {json_path}: {e}. Skipping.")
        return

    # 레이블 변환 (img_file.stem를 인자로 전달)
    yolo_label = convert_label(json_path, img_file.stem, img_width, img_height)
    
    # yolo_label이 비어있으면 (즉, 매핑 실패나 파일명 문제 발생 시) 저장하지 않음
    if not yolo_label:
        return

    # 레이블 저장 (YOLO 형식)
    label_dir = yolo_root / "labels" / target_type
    txt_path = label_dir / f"{img_file.stem}.txt"
    
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(yolo_label)

# 6. 데이터 처리 실행
process_dataset("Training", ["train"]) # 훈련 데이터는 모두 train으로
process_dataset("Validation", ["val", "test"]) # 검증 데이터는 val과 test로 분할

# 7. dataset.yaml 생성
yaml_content = {
    'path': str(yolo_root.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test', # test 경로 추가
    'names': { # names 섹션이 9개 클래스로 수정됨
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과 정상',
        4: '사과갈색무늬병',
        5: '사과과수화상병',
        6: '사과부란병',
        7: '사과점무늬낙엽병',
        8: '사과탄저병'
    }
}

with open(yolo_root / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

# 8. 분할 결과 통계 출력
def print_stats():
    print("\nDataset split statistics:")
    for split in ['train', 'val', 'test']:
        img_count = len(list((yolo_root / "images" / split).glob("*.jpg")))
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        print(f"  {split}: {img_count} images, {label_count} labels")

print("\nDataset conversion completed successfully!")
print(f"YOLO dataset structure created at: {yolo_root}")
print(f"Classes mapping: {yaml_content['names']}")
print_stats()